## Эксперимент 4_10k

In [ ]:
# !rm -rf /content/tesstrain #так удалять папку, еслли нужно

In [ ]:
!apt-get update
!apt-get install -y tesseract-ocr
!apt-get install -y tesseract-ocr-rus
!apt-get install -y libleptonica-dev

!pip install jiwer pandas tqdm

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
shell-init: error retrieving current directory: getcwd: cannot access paren

In [ ]:
!git clone https://github.com/tesseract-ocr/tesstrain
%cd tesstrain

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: could not create work tree dir 'tesstrain': No such file or directory
[Errno 2] No such file or directory: 'tesstrain'
/content/tesstrain


In [ ]:
import os

os.makedirs("/content/drive/MyDrive/OCR_models", exist_ok=True)
os.makedirs("/content/drive/MyDrive/OCR_results", exist_ok=True)

In [ ]:
import subprocess
import pandas as pd
from jiwer import cer, wer
from tqdm import tqdm

def evaluate_ocr(img_dir, gt_dir, model, output_csv):

    results = []
    files = os.listdir(img_dir)

    for img in tqdm(files):

        img_path = os.path.join(img_dir, img)
        out_path = "/content/tmp_out"

        subprocess.run([
            "tesseract",
            img_path,
            out_path,
            "-l",
            model,
            "--psm",
            "7"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        with open(out_path + ".txt") as f:
            pred = f.read().strip()

        with open(os.path.join(gt_dir, img.replace(".jpg",".gt.txt"))) as f:
            gt = f.read().strip()

        gt = " ".join(gt.split())
        pred = " ".join(pred.split())

        results.append({
            "file": img,
            "gt": gt,
            "prediction": pred,
            "cer": cer(gt, pred),
            "wer": wer(gt, pred)
        })

    df = pd.DataFrame(results)

    df.to_csv(output_csv, index=False)

    print("Saved:", output_csv)
    print("CER:", df["cer"].mean())
    print("WER:", df["wer"].mean())

    return df

In [ ]:
import os, shutil, random
from tqdm import tqdm

train_src_img = "/content/combined_dataset/train/images"
train_src_gt  = "/content/combined_dataset/train/gt"

valid_src_img = "/content/combined_dataset/valid/images"
valid_src_gt  = "/content/combined_dataset/valid/gt"

train_dst = "/content/tesstrain/data/rus_combined-ground-truth"
valid_dst = "/content/tesstrain/data/rus_combined-eval"

os.makedirs(train_dst, exist_ok=True)
os.makedirs(valid_dst, exist_ok=True)

random.seed(42)

files = [f for f in os.listdir(train_src_img) if f.endswith(".jpg")]
files = random.sample(files, 10000)

for f in tqdm(files):

    shutil.copy(
        os.path.join(train_src_img, f),
        os.path.join(train_dst, f)
    )

    shutil.copy(
        os.path.join(train_src_gt, f.replace(".jpg",".gt.txt")),
        os.path.join(train_dst, f.replace(".jpg",".gt.txt"))
    )

print("Copy validation data")

for f in tqdm(os.listdir(valid_src_img)):

    shutil.copy(
        os.path.join(valid_src_img, f),
        os.path.join(valid_dst, f)
    )

    shutil.copy(
        os.path.join(valid_src_gt, f.replace(".jpg",".gt.txt")),
        os.path.join(valid_dst, f.replace(".jpg",".gt.txt"))
    )

100%|██████████| 10000/10000 [00:04<00:00, 2114.32it/s]


Copy validation data


100%|██████████| 5000/5000 [00:01<00:00, 3675.00it/s]


In [ ]:
!mkdir -p /content/tesstrain/tessdata

!wget https://github.com/tesseract-ocr/tessdata_best/raw/main/rus.traineddata \
-P /content/tesstrain/tessdata

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
--2026-03-18 19:37:56--  https://github.com/tesseract-ocr/tessdata_best/raw/main/rus.traineddata
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/tesseract-ocr/tessdata_best/main/rus.traineddata [following]
--2026-03-18 19:37:57--  https://raw.githubusercontent.com/tesseract-ocr/tessdata_best/main/rus.traineddata
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15301764

In [ ]:
%cd /content/tesstrain

/content/tesstrain


In [ ]:
from PIL import Image
import os
from tqdm import tqdm

def convert_and_fix(folder):

    converted = 0

    for f in tqdm(os.listdir(folder)):

        path = os.path.join(folder, f)

        if f.endswith(".jpg"):

            img = Image.open(path)
            png_path = path.replace(".jpg", ".png")

            img.save(png_path, dpi=(300,300))
            os.remove(path)

            converted += 1

        elif f.endswith(".png"):

            img = Image.open(path)
            img.save(path, dpi=(300,300))

    print("converted jpg → png:", converted)

convert_and_fix("/content/tesstrain/data/rus_combined-ground-truth")
convert_and_fix("/content/tesstrain/data/rus_combined-eval")

100%|██████████| 20000/20000 [01:57<00:00, 170.67it/s]


converted jpg → png: 10000


100%|██████████| 10000/10000 [00:55<00:00, 181.72it/s]

converted jpg → png: 5000


In [ ]:
import shutil
import os
import time
import threading

src_dir = "/content/tesstrain/data"
dst_dir = "/content/drive/MyDrive/OCR_models/checkpoints"

os.makedirs(dst_dir, exist_ok=True)

def autosave():
    while True:
        for f in os.listdir(src_dir):
            if "rus_ady_checkpoint" in f:
                shutil.copy(
                    os.path.join(src_dir, f),
                    os.path.join(dst_dir, f)
                )
                print("Saved:", f)

        time.sleep(300)  # каждые 5 минут

thread = threading.Thread(target=autosave)
thread.daemon = True
thread.start()

In [ ]:
!rm -rf /content/tesstrain/data/rus_combined

In [ ]:
!ls /content/drive/MyDrive/OCR_models/checkpoints

In [ ]:
!ls data/rus_combined-eval | head

In [ ]:
!ls data/rus_combined-ground-truth | head
!ls data/rus_combined-ground-truth/*.lstmf | wc -l
!ls data/rus_combined-ground-truth/*.png | wc -l
!ls data/rus_combined-ground-truth/*.gt.txt | wc -l

In [ ]:
!mkdir -p /content/tesstrain/data/rus_combined-ground-truth-lstmf

!tesseract /content/tesstrain/data/rus_combined-ground-truth/Adyghe_NoOCR_balanced_sampled_000003_Mono_0_0.png \
          /content/tesstrain/data/rus_combined-ground-truth-lstmf/000003 \
          lstm.train -l rus_combined

In [ ]:
!ls /usr/share/tesseract-ocr/4.00/tessdata | grep rus_combined

In [ ]:
!traineddata /content/tesstrain/data/rus_combined.traineddata

In [ ]:
import os
os.makedirs("data/rus_ady/checkpoints", exist_ok=True)

In [ ]:
!make training \
MODEL_NAME=rus_combined \
START_MODEL=rus \
GROUND_TRUTH_DIR=/content/tesstrain/data/rus_combined-ground-truth \
EVAL_GROUND_TRUTH_DIR=/content/tesstrain/data/rus_combined-eval \
LANG_TYPE=LINE \
TESSDATA=/content/tesstrain/tessdata \
MAX_ITERATIONS=4000 \
DEBUG_INTERVAL=500 \
RATIO_TRAIN=0.9 \
NET_SPEC="[1,36,0,1 Ct3,3,16 Mp3,3 Lfys32 Lfx64 O1c]" \
SCROLLVIEW_PATH=/usr/bin/false \
2>&1 | tee /content/train_log.txt

make: *** No rule to make target 'training'.  Stop.


In [ ]:
!lstmtraining \
  --traineddata data/rus_combined/rus_combined.traineddata \
  --old_traineddata ./tessdata/rus.traineddata \
  --continue_from data/rus/rus_combined.lstm \
  --learning_rate 0.0001 \
  --model_output data/rus_combined/checkpoints/rus_kbd \
  --train_listfile data/rus_combined/list.train \
  --eval_listfile data/rus_combined/list.eval \
  --max_iterations 4000 \
  --target_error_rate 0.01 \
2>&1 | tee /content/train_log.txt

Error, model output cannot be written: No such file or directory


In [ ]:
!lstmtraining \
  --stop_training \
  --continue_from data/rus_combined/checkpoints/rus_combined_checkpoint \
  --traineddata data/rus_combined/rus_combined.traineddata \
  --model_output /content/rus_combined.traineddata

mgr_.Init(traineddata_path.c_str()):Error:Assert failed:in file ../../src/lstm/lstmtrainer.h, line 110


In [ ]:
!ls -lh /content/rus_combined.traineddata

from google.colab import files
files.download("/content/rus_combined.traineddata")

import shutil

shutil.copy(
    "/content/rus_combined.traineddata",
    "/content/drive/MyDrive/OCR_models/rus_combined.traineddata"
)

!cp /content/rus_combined.traineddata \
/usr/share/tesseract-ocr/4.00/tessdata/

In [ ]:
evaluate(
    "/content/ady_dataset/test/images",
    "/content/ady_dataset/test/gt",
    "rus_combined",
    "/content/drive/MyDrive/OCR_results/rus_combined_on_ady.csv"
)

evaluate(
    "/content/kab_dataset/test/images",
    "/content/kab_dataset/test/gt",
    "rus_combined",
    "/content/drive/MyDrive/OCR_results/rus_combined_on_kab.csv"
)

evaluate(
    "/content/combined_dataset/test/images",
    "/content/combined_dataset/test/gt",
    "rus_combined",
    "/content/drive/MyDrive/OCR_results/rus_combined_on_combined.csv"
)